In [1]:
from __future__ import annotations
from typing import Iterable, Generator, Any, TypeVar, Generic
from dataclasses import dataclass, field
import numpy as np
import json
from metasmith.hashing import KeyGenerator

In [2]:
class Node:
    NO_KEY = "_"
    def __init__(
        self,
        properties: set[str],
        parents: set[Node],
        _sig: str|None=None,
    ) -> None:
        super().__init__()
        assert isinstance(properties, set)
        assert isinstance(parents, set)
        self.properties = properties
        self.parents = parents
        self._sig = _sig
        self.hash, self.key = KeyGenerator.FromStr(self.Signature())
        # self._diffs = set()
        # self._sames = set()

    def __hash__(self) -> int:
        return self.hash

    def __eq__(self, __value: object) -> bool:
        return isinstance(__value, Node) and self.hash == __value.hash

    def __str__(self) -> str:
        return f"<{self._json_dumps(self.Pack(parents=False)['properties']).replace('"', '')}:{self.key}>"

    def __repr__(self) -> str:
        return f"{self}"

    def IsA(self, other: Node) -> bool:
        return other.properties.issubset(self.properties)

    def Signature(self):
        if self._sig is None:
            psig = ",".join(sorted(p.key for p in self.parents))
            sig = "".join(sorted(self.properties))
            _, sig = KeyGenerator.FromStr(sig)
            self._sig = f'{sig}:[{psig}]' if len(self.parents)>0 else sig
        return self._sig

    def Clone(self, properties_only: bool=False):
        clone = self.__class__(
            properties=set(self.properties),
            parents=set(p.Clone() for p in self.parents),
            _sig=None if properties_only else self._sig,
        )
        return clone

    def WithLineage(self, parents: Iterable[Node]):
        image = self.__class__(
            properties=self.properties,
            parents=set(parents),
        )
        return image

    @classmethod
    def _json_dumps(cls, d):
        return json.dumps(d, separators=(',', ':'), sort_keys=True)

    @classmethod
    def Unpack(cls, d: dict):
        NO_KEY = cls.NO_KEY
        raw_props = d["properties"]
        props = set()
        if type(raw_props) in {list, set}: # all properties didn't have keys
            for v in raw_props:
                assert type(v) not in {list, dict}
                props.add(v)
        elif isinstance(raw_props, dict):
            for k, v in raw_props.items():
                assert type(v) not in {dict}
                if k == NO_KEY:
                    assert type(v) in {list}
                    props.update(v)
                    continue

                if isinstance(v, list):
                    props.update(cls._json_dumps({k:x}) for x in v)
                else:
                    props.add(cls._json_dumps({k:v}))
        else:
            assert False, f"unexpected format [{type(raw_props)}: {raw_props}]"
        m = cls(
            properties=props,
            parents=set(),
        )
        if "parents" in d:
            m.parents = {cls.Unpack(x) for x in d["parents"]}
        if "_hash" in d:
            m.hash, m.key = d["_hash"].split("/")
            m.hash = int(m.hash)
        return m

    def Pack(self, parents=False):
        NO_KEY = self.NO_KEY
        props = sorted(list(self.properties))
        formatted_props = {}
        def _try_keyval(p: str):
            try:
                e = json.loads(p)
                if len(e)>1: return NO_KEY, p
                k, v = next(iter(e.items()))
                return k, v
            except json.JSONDecodeError:
                return NO_KEY, p
        for p in props:
            k, v = _try_keyval(p)
            formatted_props[k] = formatted_props.get(k, [])+[v]
        # collapse singletons, unless they didn't have a key
        for k in list(formatted_props.keys()):
            if k == NO_KEY: continue
            if len(formatted_props[k])==1:
                formatted_props[k] = formatted_props[k][0]
        # if all didn't have keys, just save as list
        if len(formatted_props) == 1 and NO_KEY in formatted_props:
            formatted_props = formatted_props[NO_KEY]
        d = {
            "properties": formatted_props,
        }
        if len(self.parents)>0 and parents:
            d["parents"] = [x.Pack() for x in self.parents]
        return d

# of a Transform
class Dependency(Node):
    def __init__(self, properties: set[str], parents: set[Dependency]) -> None:
        super().__init__(properties=properties, parents=set(parents))

    def __str__(self) -> str:
        return f"(D:{'-'.join(sorted(list(self.properties)))})"

# as in a free floating data type
class Endpoint(Node):
    def __init__(self, properties: set[str], parents: set[Endpoint]|None=None) -> None:
        p: set[Node] = set(parents) if parents is not None else set()
        super().__init__(properties=properties, parents=p)

class Transform:
    def __init__(self) -> None:
        super().__init__()
        self.requires: list[Dependency] = list()
        self.produces: list[Dependency] = list()
        self._update_hash()

    def __str__(self) -> str:
        def _props(d: Dependency):
            return "{"+"-".join(sorted(d.properties))+"}"
        return f"{','.join(_props(r) for r in self.requires)}->{','.join(_props(p) for p in self.produces)}"

    def __repr__(self) -> str:
        return str(self)

    def __hash__(self) -> int:
        return self.hash

    def _update_hash(self):
        self.hash, self.key = KeyGenerator.FromStr(str(self))

    def AddRequirement(self, properties: Iterable[str]|None=None, parents: set[Dependency]|None=None):
        return self._add_dependency(destination=self.requires, properties=properties, parents=parents)

    def AddProduct(self, properties: Iterable[str]|None=None, parents: set[Dependency]|None=None):
        return self._add_dependency(destination=self.produces, properties=properties, parents=parents)

    def _add_dependency(self, destination: list[Dependency], properties: Iterable[str]|None=None, parents: set[Dependency]|None=None):
        if parents is None: parents = set()
        _properties = set(properties) if properties else set()
        _dep = Dependency(properties=_properties, parents=parents)
        _parents = _dep.parents
        destination.append(_dep)
        if destination == self.requires:
            i = len(self.requires)-1
            for p in _parents:
                assert p in self.requires, f"{p} not added as a requirement"
        self._update_hash()
        return _dep

In [3]:
# maps which generated endpoints are mapped to the transform's dependencies
@dataclass
class Application:
    transform: Transform
    used: dict[Dependency, Endpoint]
    produced: dict[Dependency, Endpoint]
    score: list[float] = field(default_factory=list)
    _iteration: int = -1
    _sig: str|None = None
    _hash: int|None = None
    def Signature(self):
        if self._sig is None: 
            self._sig = self.transform.key + "".join({self.used[p].key for p in self.transform.requires})
        return self._sig
    def __hash__(self) -> int:
        if self._hash is None:
            self._hash, _ = KeyGenerator.FromStr(self.Signature())
        return self._hash
    def __eq__(self, value: object) -> bool:
        if not isinstance(value, Application): return False
        return self._hash == value._hash

@dataclass
class SolverState:
    steps: list[Application]
    production: dict[Dependency, list[Endpoint]] # product dep to produced endpoint
    have: set[Endpoint]
    candidate_transforms: set[Transform] # may not be valid, holds use count

@dataclass
class RefinerState:
    steps: list[Application]
    scores: list[float] = field(default_factory=lambda: [0.0])
    valid: bool = False
    _sig: str|None = None
    _hash: int = 0
    _iteration: int = -1
    def Signature(self):
        if self._sig is None:
            self._sig = "".join(sorted(s.Signature() for s in self.steps))
        return self._sig
    def __hash__(self) -> int:
        if self._hash is None:
            self._hash, _ = KeyGenerator.FromStr(self.Signature())
        return self._hash
    def __eq__(self, value: object) -> bool:
        if not isinstance(value, RefinerState): return False
        return self._hash == value._hash

@dataclass
class Solution:
    complete: bool
    steps: list[Application]
    _frontier: list[Application]
    _history: list[SolverState]
    _refiner_history: list[RefinerState]
    _heuristics: dict[str, dict[str, float]]
    _iterations: int
    _refiner_iterations: tuple[int, int] # found at, total expanded
    _relavent_transforms: list[Transform]
    
def solve_by_mcts(
    given: Iterable[Endpoint],
    transforms: Iterable[Transform],
    target: Transform,
    seed: int=42,
    max_iter: int=256,
    max_refine: int=64,
):
    np.random.seed(seed)
    # ---
    # monte carlo tree search

    given_tr = Transform()
    given_appl = Application(given_tr, used={}, produced={})
    for e in given:
        p = given_tr.AddProduct(properties=e.properties)
        given_appl.produced[p] = e
    def _iter_transforms():
        yield given_tr
        for tr in transforms: yield tr
        yield target
    # produced dependency to consuming transform
    product2consumer: dict[Dependency, set[Transform]] = {}
    for parent in _iter_transforms():
        for child in _iter_transforms():
            if parent == child: continue
            for p in parent.produces:
                if not any(p.IsA(c) for c in child.requires): continue
                product2consumer[p] = product2consumer.get(p, set())|{child}
    # requirement prototype of consumer
    # to production prototype of producer
    demand2product: dict[Dependency, set[Dependency]] = {}
    demand2producer: dict[Dependency, set[Transform]] = {}
    for child in _iter_transforms():
        for parent in _iter_transforms():
            if parent == child: continue
            for c in child.requires:
                found = False
                for p in parent.produces:
                    if not p.IsA(c): continue
                    demand2product[c] = demand2product.get(c, set())|{p}
                    found = True
                if found:
                    demand2producer[c] = demand2producer.get(c, set())|{parent}

    # estimate distance of nodes to target to provide guiding metric
    # filter out nodes that don't contribute to production of targets
    opportunity_scores: dict[Transform, int] = {}
    distance_scores: dict[Transform, int] = {}
    todo: list[tuple[Transform, int]] = [(target, -1)]
    while len(todo)>0:
        node, consumer_distance = todo.pop()
        dist = consumer_distance+1
        other_dist = distance_scores.get(node, -1)
        if dist>other_dist:
            distance_scores[node] = dist
        opportunity_scores[node] = opportunity_scores.get(node, 1)+dist
        for p in node.requires:
            for producer in demand2producer.get(p, []): # when tr requires a terminal endpoint that is not given
                todo.append((producer, dist))
    relavent_transforms = [tr for tr in transforms if tr in distance_scores]
    max_distance_score = max(distance_scores.values())
    
    # for k, v in demand2producer.items():
    #     print(k, v)
    # print()

    # for k in distance_scores:
    #     print(k)

    def _prune_irrelavent_values(d: dict, value_whitelist: set):
        for k, v in d.items():
            d[k] = value_whitelist.intersection(v)
        # for k in list(d):
        #     if len(d[k])==0: del d[k]
    rts = set(relavent_transforms)|{given_tr, target}
    _prune_irrelavent_values(product2consumer, rts)
    _prune_irrelavent_values(demand2producer, rts)
    rtsp = {p for t in rts for p in t.produces}
    _prune_irrelavent_values(demand2product, rtsp)

    # should not perform mutations
    def generate_applications_of_transform(
        production: dict[Dependency, list[Endpoint]],
        blacklist: set[str],
        tr: Transform,
        mock_produced: dict[Dependency, Endpoint]|None=None
    ) -> list[Application]:
        if len(tr.requires)==0:
            appl = Application(tr, used={}, produced={})
            if appl.Signature() in blacklist: return []
            appl.produced = {p:Endpoint(properties=p.properties) for p in tr.produces}
            return [appl]
        
        # if mock_produced is given, do not check for lineage,
        # return all possibilities, and use mock_produced for the new applications
        handle_lineage = mock_produced is None

        # Transforms define lineage constraints (LC) first.
        # The endpoint matched to the LC must also be used to satisfy all instances.
        # That is, if a transform specifies A via P and B via P, 
        # then endpoint P' matched to P must be used to create both A and B
        def _satisfies_lineage(e: Endpoint, p: Dependency, used: dict[Dependency, Endpoint]):
            for parent in p.parents:
                assert isinstance(parent, Dependency)
                matched = used[parent]
                # print(".   ", matched, e.parents, e)
                if matched not in e.parents: return False
            return True
        
        def _find_endpoints(p: Dependency):
            candidates: set[Endpoint] = set()
            # print("", p in demand2product)
            for product in demand2product.get(p, []):
                # print(" ", product.key, product)
                if product not in production: continue
                for e in production[product]:
                    assert e.IsA(p)
                    candidates.add(e)
            return candidates
        # for k, v in production.items():
        #     if '{"data":"OCI"}' in k.properties: continue
        #     print(k, v)

        # print("?  ", state.have)
        viable_input_sets: list[Application] = []
        matches: dict[Dependency, set[Endpoint]] = {}
        for p in tr.requires:
            candidates = _find_endpoints(p)
            # print("?  ", len(candidates), p)
            if len(candidates) == 0: return viable_input_sets # empty, for type def
            matches[p] = candidates
        # for k, v in matches.items():
            # print(" ?-  ", len(v), k, v)
        
        INITIAL_I = 0
        todo: list[tuple[int, Endpoint, dict[Dependency, Endpoint]]] = [
            (INITIAL_I, e, dict()) for e in matches[tr.requires[INITIAL_I]]
        ]
        while len(todo)>0:
            p_i, e, used = todo.pop()
            p = tr.requires[p_i]
            used = used|{p:e}
            # print("_  ", used)
            if handle_lineage and not _satisfies_lineage(e, p, used): continue
            if p_i >= len(tr.requires)-1:
                appl = Application(tr, used, {})
                if appl.Signature() in blacklist: continue
                if handle_lineage:
                    lineage: set = {ancestor for e in used.values() for ancestor in e.parents}
                    lineage.update(used.values())
                    appl.produced = {product:Endpoint(product.properties, parents=lineage) for product in tr.produces}
                else:
                    appl.produced = mock_produced
                viable_input_sets.append(appl)
                continue # at leaf (end of required dependencies)
            next_i = p_i+1
            todo += [
                (next_i, e, used) for e in matches[tr.requires[next_i]]
            ]
        return viable_input_sets
    
    @dataclass
    class MctsResult:
        complete: bool
        state: SolverState
        _frontier: list[Application]
        _history: list[SolverState]
        _iterations: int
    def mcts(max_iter: int):
        def is_solved(state: SolverState):
            last_transform = state.steps[-1].transform
            return last_transform == target

        def score_node(node: Application):
            dist = distance_scores[node.transform]
            dist = 1-dist/max_distance_score
            opportunity = opportunity_scores[node.transform]
            opportunity = 1-(1/(1+opportunity/10))
            node.score = [dist, opportunity]
            return node

        def select_node(frontier: list[Application]):
            probs = [75, 20, 5] # dist, opportunity, explore
            total_prob = sum(probs)
            probs = [x/total_prob for x in probs]
            p_i = np.random.choice(list(range(len(probs))), 1, p=probs)[0]
            if p_i<len(probs)-1: # exploit
                scores = np.array([s.score[p_i] for s in frontier])
                K = 1
                k = min(K, scores.shape[0])
                candidate_indexes = np.argpartition(scores, -k)[-k:]
                i: int = np.random.choice(candidate_indexes)
            else: # explore
                i = np.random.randint(0, len(frontier))
            return i

        def remove_node(frontier: list[Application], index: int):
            frontier[index], frontier[-1] = frontier[-1], frontier[index]
            return frontier.pop() # O(1) vs O(m) for arr.remove()

        def expand_node(state: SolverState, appl: Application):
            candidate_transforms = state.candidate_transforms.copy() # was free transform
            for p in appl.transform.produces:
                if p not in product2consumer: continue
                for linked in product2consumer[p]:
                    candidate_transforms.add(linked)
            production = state.production.copy()
            for p, e in appl.produced.items():
                production[p] = production.get(p, [])+[e]
            return SolverState(
                steps=state.steps+[appl],
                have=state.have|set(appl.produced.values()),
                candidate_transforms=candidate_transforms,
                production=production,
            )

        free_transforms = [t for t in relavent_transforms if len(t.requires)==0]
        def generate_child_nodes(state: SolverState):
            def _iter_transforms():
                for tr in state.candidate_transforms:
                    yield tr
                for tr in free_transforms:
                    yield tr
            for tr in _iter_transforms():
                # print("$ ", tr)
                for appl in generate_applications_of_transform(state.production, frontier_signatures, tr):
                    yield appl

        current_state = SolverState(
            steps=[],
            production={},
            have=set(),
            candidate_transforms=set(),
        )
        start = score_node(given_appl)
        frontier: list[Application] = [start]
        frontier_signatures: set[str] = {s.Signature() for s in frontier}
        history: list[SolverState] = []
        i: int = 0
        while len(frontier)>0 and i < max_iter:
            i += 1
            nodei = select_node(frontier)
            node = remove_node(frontier, nodei)
            node._iteration = i
            current_state = expand_node(current_state, node)
            # print(i, f"[{len(frontier)}]", node.transform)
            # for k in current_state.candidate_transforms:
            #     print("-", k)
            history.append(current_state)
            if is_solved(current_state):
                return MctsResult(
                    complete=True,
                    state=current_state,
                    _frontier=frontier,
                    _history=history,
                    _iterations=i,
                )
            applied_transforms: set[Transform] = set()
            for child in generate_child_nodes(current_state):
                # print(f"c", child.transform)
                child = score_node(child)
                child._iteration = -i
                frontier_signatures.add(child.Signature())
                applied_transforms.add(child.transform)
                frontier.append(child)
            current_state.candidate_transforms -= applied_transforms # all possibilities per tr explored
            # for s in frontier:
            #     print(f"f", s.transform)
            # print()

        return MctsResult(
            complete=False,
            state=current_state,
            _frontier=frontier,
            _history=history,
            _iterations=i,
        )

    # ---
    # prune spurious nodes, assumes last step is target
    def prune_steps(steps: list[Application]) -> list[Application]:
        e2source: dict[Endpoint, Application] = {}
        for step in steps:
            for e in step.produced.values():
                e2source[e] = step
    
        @dataclass
        class PruneNode:
            ref: Application|Endpoint

            def GetKey(self):
                if isinstance(self.ref, Application):
                    return self.ref.Signature()
                else:
                    return self.ref.key
                
            def GetChildren(self):
                if isinstance(self.ref, Application):
                    for x in self.ref.used.values():
                        yield x
                else:
                    if self.ref not in e2source: return
                    appl = e2source[self.ref]
                    yield appl

        start = PruneNode(steps[-1]) # last should be target
        todo: list[PruneNode] = [start]
        seen: dict[str, PruneNode] = {}
        while len(todo)>0:
            node = todo.pop(0)
            key = node.GetKey()
            if key in seen: continue
            seen[key] = node
            for x in node.GetChildren():
                todo.append(PruneNode(x))
        required = [x.ref for x in seen.values() if isinstance(x.ref, Application)]
        required.reverse()
        return required
    
    # ---
    # order nodes by steps to create
    def get_order(steps: list[Application]):
        seen: set[str] = set()
        _have: set[Endpoint] = {e for e in given}
        order: dict[str, int] = {e.key:0 for e in _have}
        while len(seen)<len(steps):
            reachable: set[Application] = set()
            # find and process separately to ensure 1 layer at a time 
            for step in steps:
                if step.Signature() in seen: continue
                if any(e not in _have for e in step.used.values()): continue
                seen.add(step.Signature())
                reachable.add(step)
            if len(reachable)==0: break # shouldn't happen/needed, but here to prevent endless loop
            for step in reachable:
                if len(step.used)>0:
                    step_depth = max(order[e.key] for e in step.used.values())+1
                else:
                    step_depth = 1
                order[step.Signature()] = step_depth
                for e in step.produced.values():
                    if e in order: continue
                    order[e.key] = step_depth+1
                _have |= {e for e in step.produced.values()}
        max_depth = max(order.values())+1
        for step in steps:
            k = step.Signature()
            if k in order: continue
            order[k] = max_depth
        return order
    
    def order_steps(order: dict[str, int], steps: list[Application]):
        return sorted(steps, key=lambda s: order[s.Signature()]*10000+len(s.used))
    
    solution = mcts(
        max_iter=max_iter
    )
    D2T_KEY = "distance to target"
    d2t_report = {k.key:float(v) for k, v in distance_scores.items()}
    if not solution.complete:
        return Solution(
            complete=False,
            steps=[],
            _frontier=solution._frontier,
            _history=solution._history,
            _refiner_history=[],
            _heuristics={
                D2T_KEY: d2t_report,
            },
            _iterations=solution._iterations,
            _refiner_iterations=0,
            _relavent_transforms=relavent_transforms,
        )

    pruned_steps = prune_steps(solution.state.steps)

    @dataclass
    class RefinerResult:
        steps: list[Application]
        _history: list[RefinerState]
        _iterations: int
        _found_on: int
    def refine_mcts(initial_solution: list[Application], max_iters: int):
        def validate_node(state: RefinerState):
            produced_from: dict[Endpoint, list[Endpoint]] = {}
            for appl in state.steps:
                _from = list(appl.used.values())
                for e in appl.produced.values():
                    produced_from[e] = _from
            def _has_ancestor(e: Endpoint, a: Endpoint):
                todo = [e]
                seen = {e}
                while len(todo)>0:
                    e = todo.pop()
                    if e == a: return True
                    for parent in produced_from[e]:
                        if parent in seen: continue
                        todo.append(parent)
                        seen.add(parent)

            def _iter_steps():
                yield given_appl
                for step in state.steps:
                    yield step
            # checks lineage constaint and no loops
            def _is_valid():
                e2appl: dict[Endpoint, list[Application]] = {}
                for appl in _iter_steps():
                    for e in appl.used.values():
                        e2appl[e] = e2appl.get(e, [])+[appl]
                todo = [(given_appl, set())]
                while len(todo)>0:
                    current, history = todo.pop()
                    if current.Signature() in history: return False # looped
                    history = history|{current.Signature()}
                    for e in current.produced.values():
                        for appl in e2appl.get(e, []):
                            todo.append((appl, history))
                # if here, then no loops
                # now check lineage
                for step in _iter_steps():
                    for p, e in step.used.items():
                        for pproto in p.parents:
                            lineage_constraint_e = step.used[pproto] # type: ignore
                            if not _has_ancestor(e, lineage_constraint_e): return False
                return True
            state.valid = _is_valid()
                
        def score_node(state: RefinerState):
            validate_node(state)
            used_as_lineage: set[Endpoint] = set()
            for step in state.steps:
                for p in step.used.keys():
                    used_as_lineage |= {step.used[pproto] for pproto in p.parents} # type: ignore
            _steps = state.steps
            lineage_usage: dict[Endpoint, int] = {}
            for step in _steps:
                for p, e in step.used.items():
                    if not e in used_as_lineage: continue
                    lineage_usage[e] = lineage_usage.get(e, 0)+1
            def _entropy(a) -> float:
                a = np.array(a)
                p = a/a.sum()
                p = p[p>0]
                return float((p*np.log2(p)).sum())
            e_score = _entropy(list(lineage_usage.values()))

            _product2producer: dict[Endpoint, Application] = {}
            for step in _steps:
                for e in step.produced.values():
                    _product2producer[e] = step
            def _max_distance_to(e: Endpoint, a: Endpoint):
                todo = [(e, 0)]
                seen = set()
                max_d = -1
                while len(todo)>0:
                    n, d = todo.pop()
                    if n in seen: continue
                    seen.add(n)
                    if n == a:
                        max_d = max(max_d, d)
                    prod = _product2producer[n]
                    for pe in prod.used.values():
                        todo.append((pe, d+1))
                return max_d/len(_steps) if max_d>0 else 1.0
            lin_distances: list[float] = []
            for step in _steps:
                for p in step.transform.requires:
                    for lin_p in p.parents:
                        e = step.used[p]
                        pe= step.used[lin_p] # type: ignore
                        lin_distances.append(_max_distance_to(e, pe))
            lin_score = -sum(lin_distances)/len(lin_distances) if len(lin_distances)>0 else 0
            score = e_score*1000+lin_score
            _, k = KeyGenerator.FromStr(state.Signature(), l=4)
            vscore = score*state.valid
            state.scores = [score, vscore]
        
        def select_node(frontier: list[RefinerState]) -> int:
            probs = [75, 20, 5] # score, score * valid
            total_prob = sum(probs)
            probs = [x/total_prob for x in probs]
            p_i = np.random.choice(list(range(len(probs))), 1, p=probs)[0]
            if p_i<len(probs)-1: # exploit
                scores = np.array([s.scores[p_i] for s in frontier])
                K = 1
                k = min(K, scores.shape[0])
                candidate_indexes = np.argpartition(scores, -k)[-k:]
                i: int = np.random.choice(candidate_indexes)
            else: # explore
                i = np.random.randint(0, len(frontier))
            return i
        
        def remove_node(frontier: list[RefinerState], index: int):
            frontier[index], frontier[-1] = frontier[-1], frontier[index]
            return frontier.pop() # O(1) vs O(m) for arr.remove()

        def expand_node(state: RefinerState):
            current_applications = {s.Signature() for s in state.steps}
            production: dict[Dependency, list[Endpoint]] = {}
            for step in state.steps:
                for p, e in step.produced.items():
                    production[p] = production.get(p, [])+[e]
            for step in state.steps:
                # reuse the current endpoints and simply look for alternate edge comparisons
                # lineage constraint checked separately
                for appl in generate_applications_of_transform(
                    production, current_applications,
                    step.transform,
                    mock_produced=step.produced, # rectify later
                ):
                    appl._iteration = step._iteration
                    alt_sol = [s for s in state.steps if s.Signature() != step.Signature()]+[appl]
                    alt_state = RefinerState(steps=alt_sol)
                    yield alt_state
        
        # produce new set of endpoints so hashes are valid
        # and prune steps
        def rectify(solution: list[Application]):
            steps = [
                Application(
                    transform=step.transform,
                    used=step.used.copy(),
                    produced=step.produced.copy(),
                    score=step.score,
                    _iteration=step._iteration,
                ) for step in [given_appl]+solution
            ]

            # prune steps, place target step last, as required
            _targeti = -1
            for i, s in enumerate(steps):
                if len(s.produced) == 0:
                    _targeti = i
                    break
            assert _targeti >= 0
            steps[_targeti], steps[-1] = steps[-1], steps[_targeti]
            steps = prune_steps(steps) # may be dangerous, since endpoint hashes are not yet fixed

            _product2consumer: dict[Endpoint, list[Application]] = {}
            for step in steps:
                for e in step.used.values():
                    _product2consumer[e] = _product2consumer.get(e, [])+[step]

            endpoint_map: dict[Endpoint, Endpoint] = {}
            rev_emap: dict[Endpoint, Endpoint] = {}
            def _fix_endpoints(appl: Application):
                lineage: set[Endpoint] = set()
                for p in appl.transform.requires:
                    e = appl.used[p]
                    e = endpoint_map.get(e, e)
                    appl.used[p] = e # update to new endpoint
                    lineage.add(e)
                    lineage.update(e.parents) # type: ignore
                # fix lineage of endpoints
                for p in appl.transform.produces:
                    e = appl.produced[p]
                    new_e = Endpoint(e.properties, parents=lineage)
                    appl.produced[p] = new_e
                    endpoint_map[e] = new_e
                    rev_emap[new_e] = e
                # force regenerate signature
                appl._sig = None
                appl._hash = None
            
            node_order = get_order(steps)
            todo: list[Application] = steps.copy()
            order = [node_order[s.Signature()] for s in todo]
            while len(todo)>0:
                si: int = np.argpartition(order, 0)[0]
                todo[si], todo[-1] = todo[-1], todo[si]
                order[si], order[-1] = order[-1], order[si]
                order.pop()
                appl = todo.pop()
                _fix_endpoints(appl) # mutates appl
            return steps

        initial_state = RefinerState(
            steps=initial_solution,
            valid=True,
        )
        score_node(initial_state)
        frontier: list[RefinerState] = [initial_state]
        seen: set[str] = {initial_state.Signature()}
        valids: list[RefinerState] = [initial_state]
        history: list[RefinerState] = []
        i = 0
        while len(frontier)>0 and i<max_iters:
            i += 1
            statei = select_node(frontier)
            state = remove_node(frontier, statei)
            state._iteration = i
            history.append(state)
            if state.valid:
                valids.append(state)
            for child in expand_node(state):
                if child.Signature() in seen: continue
                seen.add(child.Signature())
                score_node(child)
                frontier.append(child)
        scores = np.array([s.scores[1] for s in valids]) # take the valid score
        k = 1
        si: int = np.argpartition(scores, -k)[-k:][0]
        refined = valids[si]
        return RefinerResult(
            steps=rectify(refined.steps),
            _history=history,
            _iterations=i,
            _found_on=refined._iteration,
        )

    refined = refine_mcts(pruned_steps, max_refine)
    _steps = refined.steps
    node_order = get_order(_steps)
    ordered_steps = order_steps(node_order, _steps)

    return Solution(
        complete=True,
        steps=ordered_steps,
        _frontier=solution._frontier,
        _history=solution._history,
        _refiner_history=refined._history,
        _heuristics={
            "production depth": {k:float(v) for k, v in node_order.items()},
            D2T_KEY: d2t_report,
        },
        _iterations=solution._iterations,
        _refiner_iterations=(refined._found_on, refined._iterations),
        _relavent_transforms=relavent_transforms,
    )

In [4]:
from enum import Enum
from pathlib import Path
import re
from metasmith.hashing import KeyGenerator

def _clean(s:str):
    return re.sub(r'[\{\}\"]', "", s)

class DAGRenderer:
    def __init__(self, plan: list[Application], meta: dict[str, Any]|None=None) -> None:
        self.plan = plan
        self.meta = meta if meta is not None else {}

    class NodeType(Enum):
        TRANSFORM = 1
        DATA      = 2
    def RenderNode(self, **meta) -> str:
        style: dict[str, str] = {}
        ntype = meta["ntype"]
        name = meta["name"]
        for k in ["style", "color"]:
            if k not in meta: continue
            style[k] = meta[k]
        match ntype:
            case self.NodeType.TRANSFORM:
                style["shape"] = "oval"
            case self.NodeType.DATA:
                style["shape"] = "box"
        style_str = " ".join(f'{k}="{v}"' for k, v in style.items())
        return f'"{name}" [{style_str}]'

    def AsDAG(self, *, font: str = 'Arial', hide_images: bool = True) -> str:
        lines = ["digraph G {"]
        lines += [f'graph [fontname="{font}"];', f'node  [fontname="{font}"];', f'edge  [fontname="{font}"];']
        plan = self.plan
        for step in plan:
            k = step.Signature()
            transform_name = f"{step.transform}:{KeyGenerator.FromStr(k, l=4)[1]}"
            meta = self.meta.get(k, {})
            if "name" in meta:
                transform_name = meta['name']
            if "prefix" in meta:
                transform_name = f"{meta['prefix']}:{transform_name}"
            transform_name = _clean(transform_name)
            meta['name'] = transform_name
            lines.append(self.RenderNode(ntype=self.NodeType.TRANSFORM, **meta))
            data_nodes = [("i", p, e) for p, e in step.used.items()]
            data_nodes += [("o", p, e) for p, e in step.produced.items()]
            for direction, p, e in data_nodes:
                meta = self.meta.get(str(e), {})
                if "name" not in meta:
                    name = _clean(str(e))
                    meta["name"] = name
                else:
                    name = meta["name"]
                lines.append(self.RenderNode(ntype=self.NodeType.DATA, **meta))
                match direction:
                    case "i":
                        lines.append(f'    "{name}" -> "{transform_name}";')
                    case "o":
                        lines.append(f'    "{transform_name}" -> "{name}";')
        lines.append("}")
        return "\n".join(lines)

    def RenderDAG(self, path_base: Path|str, format: str ='svg', *, font: str = 'Arial', hide_images: bool = True):
        import graphviz
        dag_str = self.AsDAG(font=font, hide_images=hide_images)
        src = graphviz.Source(dag_str, filename=path_base, format=format)
        src.render(cleanup=True)

In [5]:
def _tr(uses, makes):
    t = Transform()
    for x in uses:
        t.AddRequirement(properties={p.strip() for p in x.split(",")})
    for x in makes:
        t.AddProduct(properties={p.strip() for p in x.split(",")})
    return t

# ts: list[Transform] = []
# ts.append(_tr(uses = ["d0-0"], makes = ["d1-0"]))
# ts.append(_tr(uses = ["d1-0"], makes = ["d2-0"]))

# given = [
#     Endpoint(properties={"d0-0"}),
# ]

# target = Transform()
# target.AddRequirement(properties={"d2-0"})

ts: list[Transform] = []
ts.append(_tr(uses = ["start"], makes = ["a", "b"]))
ts.append(_tr(uses = ["a"], makes = ["via, v1"]))
ts.append(_tr(uses = ["b"], makes = ["via, v2"]))
ts.append(_tr(uses = ["via"], makes = ["target"]))

given = [
    Endpoint(properties={"start"}),
]

target = Transform()
p = target.AddRequirement(properties={"via"})
target.AddRequirement(properties={"target"}, parents={p})

solution = solve_by_mcts(given, ts, target)
if solution.complete:
    print(f"steps: {len(solution.steps)}")
    order = solution._heuristics["production depth"]
    for appl in solution.steps:
        depth = order[appl.Signature()]
        depth = int((depth-1)/2)
        print(depth, appl.transform)
    renderer = DAGRenderer(solution.steps)
    renderer.RenderDAG("./cache/mcts", format="png")
import os
from tqdm import tqdm
Path("./cache/hist").mkdir(exist_ok=True, parents=True)
os.system(f"rm ./cache/hist/*")
print(f"expanded [{len(solution._history)}], refiner [{len(solution._refiner_history)}]")
todo = solution._history[:50]
for i, picked in tqdm(enumerate(todo), total=len(todo)):
    _depths = solution._heuristics["distance to target"]
    l = len(picked.steps)
    d = {a.Signature():{
        # "prefix":f"{i}/{int(_depths[a.transform.key])}/{a.iteration}",
        "prefix":f"{a._iteration}",
        "style":f"filled",
        # "color":f"0,0,{0.5+((l-i)/(2*l))}",
        "color":f"0,0,{1-0.5*(a._iteration/solution._iterations)}",
    } for i, a in enumerate(picked.steps)}
    renderer = DAGRenderer(picked.steps, d)
    renderer.RenderDAG(f"./cache/hist/{i:05}", format="png")

steps: 5
0 ->{start}
1 {start}->{a},{b}
2 {a}->{v1-via}
3 {via}->{target}
4 {via},{target}->
expanded [5], refiner [1]


100%|██████████| 5/5 [00:00<00:00, 40.75it/s]


In [6]:
ts: list[Transform] = []

ts.append(_tr(uses = ["lr accession"], makes = ["lr"]))
ts.append(_tr(uses = ["lr"], makes = ["lr filtered"]))
ts.append(_tr(uses = ["lr"], makes = ["lr self map"]))
ts.append(_tr(uses = ["lr self map"], makes = ["miniasm est"]))

t = Transform()
via = t.AddRequirement(properties={"lr"})
t.AddRequirement(properties={"miniasm est"}, parents={via})
t.AddProduct(properties={"lr filtered"})
ts.append(t)

# ts.append(_tr(uses = ["lr filtered"], makes = ["assembly, lr asm"]))
t = Transform()
via = t.AddRequirement(properties={"miniasm est"})
t.AddRequirement(properties={"lr filtered"}, parents={via})
t.AddProduct(properties={"assembly", "lr asm"})
ts.append(t)

ts.append(_tr(uses = ["sr accession"], makes = ["sr"]))
ts.append(_tr(uses = ["sr"], makes = ["sr trimmed"]))
ts.append(_tr(uses = ["sr trimmed"], makes = ["assembly, sr asm"]))

ts.append(_tr(uses = ["sr trimmed", "assembly, lr asm"], makes = ["seq aln map"]))
ts.append(_tr(uses = ["seq aln map"], makes = ["bin aln map"]))
ts.append(_tr(uses = ["bin aln map", "assembly, lr asm"], makes = ["assembly, hybrid asm"]))

ts.append(_tr(uses = ["assembly"], makes = ["genes", "cds"]))

ts.append(_tr(uses = [], makes = ["bakta ref"]))
ts.append(_tr(uses = [], makes = ["busco ref", "busco map"]))
ts.append(_tr(uses = [], makes = ["cazy ref"]))
ts.append(_tr(uses = [], makes = ["kfs ko_list"]))
ts.append(_tr(uses = [], makes = ["kfs profiles"]))

t = Transform()
asm = t.AddRequirement(properties={"assembly"})
t.AddRequirement(properties={"bakta ref"})
t.AddRequirement(properties={"cds"}, parents={asm})
t.AddRequirement(properties={"genes"}, parents={asm})
t.AddProduct(properties={"bakta ann"})
ts.append(t)

ts.append(_tr(uses = ["cazy ref", "cds"], makes = ["cazy ann"]))
ts.append(_tr(uses = ["busco ref", "busco map", "cds"], makes = ["busco ann"]))
ts.append(_tr(uses = ["kfs profiles", "kfs ko_list", "cds"], makes = ["kfs ann"]))

t = Transform()
asm = t.AddRequirement(properties={"assembly"})
t.AddRequirement(properties={"bakta ann"}, parents={asm})
t.AddRequirement(properties={"cazy ann"}, parents={asm})
t.AddRequirement(properties={"busco ann"}, parents={asm})
t.AddRequirement(properties={"kfs ann"}, parents={asm})
t.AddProduct(properties={"annotations"})
ts.append(t)

t = Transform()
asm = t.AddRequirement(properties={"assembly"})
t.AddRequirement(properties={"annotations"}, parents={asm})
t.AddProduct(properties={"figures"})
ts.append(t)

given = [
    Endpoint(properties={"lr accession"}),
    Endpoint(properties={"sr accession"}),
    # Endpoint(properties={"sr"}),
    # Endpoint(properties={"assembly", "hybrid asm"}),
    # Endpoint(properties={"assembly", "lr asm"}),
    # Endpoint(properties={"bin aln map"}),
]

target = Transform()
# p = target.AddRequirement(properties={"assembly", "sr asm"})
# p = target.AddRequirement(properties={"miniasm est"})
# p = target.AddRequirement(properties={"assembly", "lr asm"}, parents={p})
# p = target.AddRequirement(properties={"assembly", "lr asm"})
p = target.AddRequirement(properties={"assembly", "hybrid asm"})
# target.AddRequirement(properties={"genes"})
# target.AddRequirement(properties={"genes"}, parents={p})
# target.AddRequirement(properties={"bakta ann"})
# target.AddRequirement(properties={"bakta ann"}, parents={p})
# target.AddRequirement(properties={"cazy ann"}, parents={p})
# target.AddRequirement(properties={"kfs ann"})
# target.AddRequirement(properties={"kfs ann"}, parents={p})
# target.AddRequirement(properties={"annotations"})
target.AddRequirement(properties={"annotations"}, parents={p})
# target.AddRequirement(properties={"figures"})

# %prun solution = solve_by_mcts(given, ts, target, max_iter=2**12)
solution = solve_by_mcts(given, ts, target, max_refine=2**8, seed=1)
if solution.complete:
    print(f"iters: {solution._iterations}, solution size: {len(solution.steps)}, refiner: {solution._refiner_iterations}")
    order = solution._heuristics["production depth"]
    d = {}
    l = len(solution.steps)
    for i, appl in enumerate(solution.steps):
        k = appl.transform.key
        k = appl.Signature()
        depth = order[k]
        depth = int((depth-1)/2)
        d[k] = {
            # "prefix": f"{depth}",
            "prefix":f"{appl._iteration}",
            # "color": f"0,0,{0.5+((l-i)/(2*l))}",
            "color": f"0,0,{0.5+0.5*((solution._iterations-appl._iteration)/solution._iterations)}",
            "style": "filled",
        }
        print(depth, appl.transform)
    renderer = DAGRenderer(solution.steps, d)
    renderer.RenderDAG("./cache/mcts")
    renderer.RenderDAG("./cache/mcts", format="png")
else:
    print(len(solution._frontier), solution._iterations)

iters: 36, solution size: 23, refiner: (15, 128)
0 ->{lr accession},{sr accession}
0 ->{kfs ko_list}
0 ->{kfs profiles}
0 ->{busco ref},{busco map}
0 ->{cazy ref}
0 ->{bakta ref}
0 {sr accession}->{sr}
0 {lr accession}->{lr}
1 {lr}->{lr self map}
1 {sr}->{sr trimmed}
2 {lr self map}->{miniasm est}
3 {lr},{miniasm est}->{lr filtered}
4 {miniasm est},{lr filtered}->{assembly-lr asm}
5 {sr trimmed},{assembly-lr asm}->{seq aln map}
6 {seq aln map}->{bin aln map}
7 {bin aln map},{assembly-lr asm}->{assembly-hybrid asm}
8 {assembly}->{genes},{cds}
9 {cazy ref},{cds}->{cazy ann}
9 {kfs profiles},{kfs ko_list},{cds}->{kfs ann}
9 {busco ref},{busco map},{cds}->{busco ann}
9 {assembly},{bakta ref},{cds},{genes}->{bakta ann}
10 {assembly},{bakta ann},{cazy ann},{busco ann},{kfs ann}->{annotations}
11 {assembly-hybrid asm},{annotations}->


In [7]:
from pathlib import Path
from metasmith.python_api import Agent, Source, Std, DataInstanceLibrary

std_dtypes, std_containers, std_transforms = Std()

In [8]:
dp2name = {}
for name, e in std_dtypes:
    print(name, e)
    k = "".join(sorted(e.properties))
    dp2name[k] = name

short_reads <{data:Short sequence,format:Sequence file}:M37WupEI>
long_reads <{data:Long sequence,format:Sequence file}:U6UcLaaP>
self_mappings <{format:Self-to-self mappings with minimap2}:ikG9D3QT>
miniasm_estimate <{format:Target bases estimated by miniasm}:FtqgfwUd>
read_stats <{data:Read statistics,format:Directory}:3BRou2gN>
short_reads_accession <{data:Accession number associated with short reads,format:Plaintext file}:4pOv1Zyo>
long_reads_accession <{data:Accession number associated with long reads,format:Plaintext file}:4HMb09gR>
long_reads_filtered <{data:Long reads filtered by filtlong,format:Sequence file}:DBwdPw8R>
short_reads_trimmed <{data:Short reads trimmed by Trimmomatic,format:Sequence file}:0aQPhgST>
long_reads_assembly <{data:Sequence assembly,format:Directory,from:Long reads}:5LXvuMiT>
short_reads_assembly <{data:Sequence assembly,format:Directory,from:Short reads}:yt0bPYmR>
hybrid_assembly <{data:Sequence assembly,format:Directory,from:Long reads, improved by sho

In [9]:
std_transforms_converted: list[Transform] = []
std_tnames = {}
for p, _, tr in std_transforms.IterateTransforms():
    m = Transform()
    used: dict[str, Dependency] = {}
    for d in tr.model.requires:
        parents = {used[p.key] for p in d.parents}
        newd = m.AddRequirement(d.properties, parents=parents)
        used[d.key] = newd
    for d in tr.model.produces:
        m.AddProduct(d.properties)
    # print(p.stem, m.key)
    std_tnames[m.key] = p.stem
    std_transforms_converted.append(m)
len(std_transforms_converted)

27

In [17]:
given: list[Endpoint] = []
for p, n, e in std_containers.Iterate():
    given.append(Endpoint(e.properties))
def _add(k):
    e = std_dtypes[k]
    assert e is not None
    given.append(Endpoint(e.properties))
# _add("short_reads")
# _add("long_reads")
_add("short_reads_accession")
_add("long_reads_accession")

target = Transform()
# p1 = target.AddRequirement(
#     # properties=std_dtypes["long_reads_filtered"].properties
#     # properties=std_dtypes["short_reads_assembly"].properties
#     # properties=std_dtypes["long_reads_assembly"].properties
#     # properties=std_dtypes["hybrid_assembly"].properties
# )
for s in [
    # "cazy_annotations",
    # "busco_annotations",
    # "kofamscan_annotations",
    # "bakta_annotations",
    "functional_annotations",
]:
    p2 = target.AddRequirement(
        properties=std_dtypes[s].properties,
        # parents={p1},
    )


# %prun solution = solve_by_mcts(given, std_transforms_converted, target, max_iter=2**10, max_refine=2**8, seed=1)
solution = solve_by_mcts(given, std_transforms_converted, target, max_iter=2**10, max_refine=2**8, seed=1)
if solution.complete:
    print(f"iters: {solution._iterations}, solution size: {len(solution.steps)}, refiner: {solution._refiner_iterations}")
    order = solution._heuristics["production depth"]
    d = {}
    l = len(solution.steps)
    for i, appl in enumerate(solution.steps):
        k = appl.transform.key
        k = appl.Signature()
        depth = order[k]
        depth = int((depth-1)/2)
        d[k] = {
            # "prefix": f"{depth}",
            "prefix":f"{appl._iteration}[{depth}]",
            # "color": f"0,0,{0.5+((l-i)/(2*l))}",
            "color": f"0,0,{0.5+0.5*((solution._iterations-appl._iteration)/solution._iterations)}",
            "style": "filled",
        }
        tk = appl.transform.key
        if tk in std_tnames:
            d[k]["name"] = f"{std_tnames[tk]}"
        if len(appl.transform.produces) == 0:
            d[k]["name"] = "target"
        for p, e in list(appl.used.items())+list(appl.produced.items()):
            name = dp2name["".join(sorted(e.properties))]
            d[str(e)] = dict(name=name)
        print(depth, d[k].get("name", appl.transform))
    renderer = DAGRenderer(solution.steps[1:], d)
    renderer.RenderDAG("./cache/mcts")
    renderer.RenderDAG("./cache/mcts", format="png")
else:
    print(len(solution._frontier), solution._iterations)

iters: 19, solution size: 16, refiner: (1, 1)
0 ->{{"data":"OCI"}-{"format":"Software container"}-{"provides":"fastqc"}},{{"data":"OCI"}-{"format":"Software container"}-{"provides":"longqc"}},{{"data":"OCI"}-{"format":"Software container"}-{"provides":"fasterq_dump"}},{{"data":"OCI"}-{"format":"Software container"}-{"provides":"filtlong"}},{{"data":"OCI"}-{"format":"Software container"}-{"provides":"flye"}},{{"data":"OCI"}-{"format":"Software container"}-{"provides":"trimmomatic"}},{{"data":"OCI"}-{"format":"Software container"}-{"provides":"megahit"}},{{"data":"OCI"}-{"format":"Software container"}-{"provides":"miniasm"}},{{"data":"OCI"}-{"format":"Software container"}-{"provides":"minimap2"}},{{"data":"OCI"}-{"format":"Software container"}-{"provides":"samtools"}},{{"data":"OCI"}-{"format":"Software container"}-{"provides":"pilon"}},{{"data":"OCI"}-{"format":"Software container"}-{"provides":"bakta"}},{{"data":"OCI"}-{"format":"Software container"}-{"provides":"diamond"}},{{"data":"O

In [197]:
print(len(solution._refiner_history))
x = []
for i, s in enumerate(solution._refiner_history):
    if not s.valid: continue
    x.append((s, s.scores[0]))
    # print(i, s.scores)
x = sorted(x, key=lambda a: a[1], reverse=True)
x[:10]

256


[(RefinerState(steps=[Application(transform={{"data":"Short sequence"}-{"format":"Sequence file"}},{{"data":"OCI"}-{"format":"Software container"}-{"provides":"trimmomatic"}}->{{"data":"Short reads trimmed by Trimmomatic"}-{"format":"Sequence file"}}, used={(D:{"data":"Short sequence"}-{"format":"Sequence file"}): <{data:Short sequence,format:Sequence file}:M37WupEI>, (D:{"data":"OCI"}-{"format":"Software container"}-{"provides":"trimmomatic"}): <{data:OCI,format:Software container,provides:trimmomatic}:eDXsaY6B>}, produced={(D:{"data":"Short reads trimmed by Trimmomatic"}-{"format":"Sequence file"}): <{data:Short reads trimmed by Trimmomatic,format:Sequence file}:yPaHA40D>}, score=[0.5833333333333334, 0.9333333333333333], _iteration=6, _sig='dAWbnMcNM37WupEIeDXsaY6B', _hash=None), Application(transform={{"data":"Long sequence"}-{"format":"Sequence file"}},{{"data":"OCI"}-{"format":"Software container"}-{"provides":"minimap2"}}->{{"format":"Self-to-self mappings with minimap2"}}, used=

In [202]:
for appl in solution._history[630].steps:
    tr = appl.transform
    tk = tr.key
    name = std_tnames.get(tk, f'{len(tr.requires)}:{len(tr.produces)}')
    print(f"{name} {appl.score}")
    for p, e in appl.used.items():
        ename = dp2name["".join(sorted(e.properties))]
        print(" ", f"{e.key} {ename}")
    print("->")
    for p, e in appl.produced.items():
        ename = dp2name["".join(sorted(e.properties))]
        print(" ", f"{e.key} {ename}")
    print("")


0:18 [1.0, 0.9985326485693323]
->
  Q5iZoUuF oci_image_fastqc
  5a3igX17 oci_image_longqc
  L64ygWKM oci_image_fasterq_dump
  jbt3CHkr oci_image_filtlong
  M8gkT5nr oci_image_flye
  eDXsaY6B oci_image_trimmomatic
  QGnq1Rur oci_image_megahit
  o9eSmQ5N oci_image_miniasm
  3ozMVljb oci_image_minimap2
  UdTli8Kg oci_image_samtools
  YtiKAjV9 oci_image_pilon
  G9t4jwWI oci_image_bakta
  eNiO47nk oci_image_diamond
  bSbFSawZ oci_image_kofamscan
  Wxyy353y oci_image_prodigal
  RvCGrKlx oci_image_ubuntu
  M37WupEI short_reads
  U6UcLaaP long_reads

self_mappings [0.8333333333333334, 0.9836867862969005]
  U6UcLaaP long_reads
  3ozMVljb oci_image_minimap2
->
  GnUZPvmb self_mappings

miniasm [0.75, 0.9812382739212008]
  U6UcLaaP long_reads
  GnUZPvmb self_mappings
  o9eSmQ5N oci_image_miniasm
->
  MCbGQCHc miniasm_estimate

filtlong_improved [0.6666666666666666, 0.9603174603174603]
  U6UcLaaP long_reads
  MCbGQCHc miniasm_estimate
  jbt3CHkr oci_image_filtlong
->
  BkzBPDzq long_reads_filtered

In [172]:
import os
from tqdm import tqdm
Path("./cache/hist").mkdir(exist_ok=True, parents=True)
os.system(f"rm ./cache/hist/*")
print(f"expanded [{len(solution._history)}]")
# print(f"refine checked [{len(solution._refiner_history)}]")
todo = [s for s in solution._history]
# todo = [s for s in solution._refiner_history if s.valid][17:21]
# todo = [s for s in solution._refiner_history if s.valid]
# todo = [s for s, _ in x[:5]]
start, end, skip = 73, 593, 10
for i, state in tqdm(enumerate(todo), total=len(todo)):
    if i < start: continue
    if i > end: break
    if (i-start)%skip != 0: continue
    _depths = solution._heuristics["distance to target"]
    # order = solution._heuristics["production depth"]
    d = {}
    l = len(solution.steps)
    for i, appl in enumerate(state.steps):
        k = appl.Signature()
        # depth = order[k]
        # depth = int((depth-1)/2)
        d[k] = {
            # "prefix": f"{depth}",
            "prefix":f"{appl._iteration}",
            # "color": f"0,0,{0.5+((l-i)/(2*l))}",
            "color": f"0,0,{0.5+0.5*((solution._iterations-appl._iteration)/solution._iterations)}",
            "style": "filled",
        }
        tk = appl.transform.key
        if tk in std_tnames:
            d[k]["name"] = f"{std_tnames[tk]}"
        if len(appl.transform.produces) == 0:
            d[k]["name"] = "target"
        for p, e in list(appl.used.items())+list(appl.produced.items()):
            name = dp2name["".join(sorted(e.properties))]
            d[str(e)] = dict(name=name)

    
    renderer = DAGRenderer(state.steps, d)
    # _, k = KeyGenerator.FromStr(state.Signature(), l=4)
    # renderer.RenderDAG(f"./cache/hist/{i:05}_score={state.scores[0]:0.6f}_valid={state.valid}_hash={k}", format="png")
    _, k = KeyGenerator.FromStr("".join(s.Signature() for s in state.steps), l=4)
    renderer.RenderDAG(f"./cache/hist/{i:05}_hash={k}", format="png")

rm: cannot remove './cache/hist/*': No such file or directory


expanded [631]


  0%|          | 0/631 [00:00<?, ?it/s]

2025-08-10_00-32-58 D| os.makedirs('./cache/hist')
2025-08-10_00-32-58 D| write lines to './cache/hist/00073_hash=jtWb'
2025-08-10_00-32-58 D| run [PosixPath('dot'), '-Kdot', '-Tpng', '-O', '00073_hash=jtWb']
2025-08-10_00-32-59 D| delete './cache/hist/00073_hash=jtWb'


 12%|█▏        | 74/631 [00:01<00:07, 71.64it/s]

...

2025-08-10_00-33-03 D| os.makedirs('./cache/hist')
2025-08-10_00-33-03 D| write lines to './cache/hist/00113_hash=RSFh'
2025-08-10_00-33-03 D| run [PosixPath('dot'), '-Kdot', '-Tpng', '-O', '00113_hash=RSFh']
2025-08-10_00-33-05 D| delete './cache/hist/00113_hash=RSFh'


 18%|█▊        | 114/631 [00:06<00:45, 11.33it/s]

2025-08-10_00-33-05 D| os.makedirs('./cache/hist')
2025-08-10_00-33-05 D| write lines to './cache/hist/00123_hash=gTuC'
2025-08-10_00-33-05 D| run [PosixPath('dot'), '-Kdot', '-Tpng', '-O', '00123_hash=gTuC']


 19%|█▉        | 123/631 [00:06<00:27, 18.42it/s]


KeyboardInterrupt: 

In [47]:
def _entropy(a) -> float:
    a = np.array(a)
    p = a/a.sum()
    p = p[p>0]
    return float((p*np.log2(p)).sum())
for x in [
    [],
    [1],
    [1, 1],
    [1, 5],
    [1, 1, 5],
]:
    print(_entropy(x))

0.0
0.0
-1.0
-0.6500224216483541
-1.1488348542809166
